# 📚 Notebook 2: Segmented Log (the BETTER way)

Instead of one infinite file, we keep a **series of segment files**. When the active segment hits a size threshold, we *roll* to a new one. Old segments can be deleted (or archived) with a single `os.unlink` — no rewriting.

This is how Kafka, BookKeeper, and most LSM-tree storage engines organize their on-disk logs.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/segmented-log
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟨 Implementation

In [ ]:
import os, tempfile, glob, time

class SegmentedLog:
    def __init__(self, dir_, max_segment_bytes=4096):
        self.dir = dir_
        os.makedirs(dir_, exist_ok=True)
        self.max_bytes = max_segment_bytes
        self.active_id = 0
        self.active_size = 0
        self._open_active()

    def _seg_path(self, sid):
        return os.path.join(self.dir, f'segment-{sid:08d}.log')

    def _open_active(self):
        self.fp = open(self._seg_path(self.active_id), 'ab')
        self.active_size = self.fp.tell()

    def append(self, record: bytes):
        framed = len(record).to_bytes(4,'big') + record
        if self.active_size + len(framed) > self.max_bytes:
            self.roll()
        self.fp.write(framed)
        self.active_size += len(framed)

    def roll(self):
        self.fp.close()
        self.active_id += 1
        self._open_active()
        print(f'  ↪ rolled to segment {self.active_id}')

    def segments(self):
        return sorted(glob.glob(os.path.join(self.dir, 'segment-*.log')))

    def delete_segments_before(self, sid):
        for p in self.segments():
            sid_in_path = int(p.split('-')[-1].split('.')[0])
            if sid_in_path < sid:
                os.unlink(p)
                print(f'  🗑 removed {os.path.basename(p)}')

WORKDIR = tempfile.mkdtemp(prefix='seglog_')
log = SegmentedLog(WORKDIR, max_segment_bytes=512)
for i in range(200):
    log.append(f'event-{i:04d}'.encode())

print('segments on disk:')
for p in log.segments():
    print(' ', os.path.basename(p), os.path.getsize(p), 'bytes')


## 🧹 Compaction / retention by segment

Dropping old data is now an `unlink` — **O(1) per segment**, regardless of how many records are inside.

In [ ]:
t0 = time.perf_counter()
log.delete_segments_before(log.active_id - 2)  # keep only the last 2 segments
print(f'compaction done in {time.perf_counter()-t0:.6f}s')
print('segments left:', len(log.segments()))


## 📊 Compare: single file vs segmented

| Concern | Single file | Segmented |
|---|---|---|
| Drop oldest data | rewrite whole file (O(n)) | unlink files (O(segments)) |
| Parallel reads | one fd contention | one fd per segment |
| Backup/ship | huge atomic file | incremental segments |
| Crash recovery scan | scan everything | scan only the active segment |

## 🚀 Best practices

- Use **time-based** *or* **size-based** rolling (whichever happens first).
- Keep a small **sparse index** per segment (offset → file position) so you can binary-search rather than scan.
- Make segment names sortable (zero-padded numeric IDs) so directory listing == log order.